# A2 · Gradient descent for linear regression

**Type**: Implement from scratch · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Implement gradient descent for OLS, show that on unscaled features it diverges, and compute the maximum stable
learning rate starting from the Hessian.

## Dataset

Synthetic, 300 rows: `area` (square meters, order of magnitude 10²) and `income` (euros, order of magnitude
10⁴), with `y = 50 + 2·area + 0.003·income + noise`. The two features have units of measure
deliberately far apart.

## Educational purpose

Connect three things that textbooks usually present separately: the OLS cost function, its gradient and
convexity. You will see that the scale of the features decides **how large the learning rate can be**, not just the speed of
convergence.

## Task

1. Write `gradient_descent(A, y, lr, n_iter)`: A already contains the column of ones; start from `w = 0`; at every iteration record the MSE in `history` and update `w ← w − lr · (2/n) · Aᵀ(Aw − y)`. If the MSE is not finite or exceeds 1e30 break the loop. Return `(w, history)` with `history` as a numpy array.
2. On the **raw** data (`A_raw`) run `lr=0.01` for 200 iterations. Assign to `diverges_raw` `True` if the last recorded MSE is greater than the first.
3. Standardize the features (subtract the mean and divide by the standard deviation) to build `A_std`, run `lr=0.1` for 500 iterations and compare `w_gd` with the exact solution `w_ols` (`np.linalg.lstsq`).
4. Compute the Hessian of the MSE `H = (2/n) AᵀA` on the standardized data and the maximum stable learning rate `lr_max = 2 / λ_max(H)`. Check that `0.9 * lr_max` converges (`conv_09`) and that `1.1 * lr_max` diverges (`div_11`).
5. Repeat the computation of `lr_max` on the raw data (`lr_max_raw`) and notice by how many orders of magnitude it is smaller.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

rng = np.random.default_rng(0)
n = 300
area = rng.normal(90, 30, n)                     # square meters
income = rng.normal(30_000, 10_000, n)           # euros
X = np.column_stack([area, income])
y = 50 + 2.0 * area + 0.003 * income + rng.normal(0, 5, n)

A_raw = np.column_stack([np.ones(n), X])

## Your work

In [ ]:
def gradient_descent(A, y, lr, n_iter):
    n = len(y)
    w = np.zeros(A.shape[1])
    history = []
    for _ in range(n_iter):
        ...
    return w, np.array(history)

In [ ]:
w_raw, history_raw = gradient_descent(A_raw, y, lr=0.01, n_iter=200)
diverges_raw = ...

In [ ]:
X_std = ...
A_std = np.column_stack([np.ones(n), X_std])
w_gd, history_std = gradient_descent(A_std, y, lr=0.1, n_iter=500)
w_ols = np.linalg.lstsq(A_std, y, rcond=None)[0]

In [ ]:
H = ...
lr_max = ...
w_a, history_a = gradient_descent(A_std, y, lr=0.9 * lr_max, n_iter=500)
w_b, history_b = gradient_descent(A_std, y, lr=1.1 * lr_max, n_iter=500)
conv_09 = ...
div_11 = ...

In [ ]:
lr_max_raw = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='The gradient is `(2 / n) * A.T @ (A @ w - y)`. The eigenvalues of a symmetric matrix are computed with `np.linalg.eigvalsh(H)`. To declare divergence compare `history[-1]` with `history[0]`, remembering that the loop stops when the MSE is no longer finite.'):
    check(isinstance(history_std, np.ndarray) and len(history_std) == 500,
          "the function returns the MSE history as an array", "`history` must be an array with one MSE per iteration")
    check(bool(diverges_raw), "on the raw data gradient descent diverges",
          "with lr=0.01 on unscaled data the MSE should explode")
    check(close(w_gd, w_ols, atol=1e-3), "on the standardized data you reach the OLS solution",
          "w_gd does not match the exact solution: check the gradient (2/n) A^T (A w - y)")
    check(np.all(np.diff(history_std) <= 1e-12), "the MSE never grows (convex function, small lr)",
          "with a small learning rate the MSE should go down at every step")
    lr_expected = 2 / np.linalg.eigvalsh((2 / n) * A_std.T @ A_std).max()
    check(close(lr_max, lr_expected, rtol=1e-6), "correct lr_max",
          "lr_max = 2 / (largest eigenvalue of H), with H = (2/n) A^T A")
    check(bool(conv_09) and bool(div_11), "0.9*lr_max converges and 1.1*lr_max diverges",
          "conv_09 must be True and div_11 True: compare the last MSE with the first")
    check(lr_max_raw < 1e-6 and lr_max / lr_max_raw > 1e6,
          "on the raw data the stable learning rate is millions of times smaller",
          "compute lr_max on the raw matrix: it should come out of the order of 1e-9")
done()

---
The commented solution is in `solutions/A_from_scratch/A2_gradient_descent.ipynb`. Open it after you pass the check, or after you have really tried.